In [7]:
import pandas as pd
from tqdm import tqdm
import pickle

## Analysis:
https://ftp.ncbi.nlm.nih.gov/refseq/release/README

In [8]:
def read(file_name):
    try:
        fasta = open(file_name, 'r')
        fasta_pairs = {}
        header = ''
        # check every line for a new header
        # if so, set as key and set non header lines as value
        for line in fasta:
            if line[0] == '>':
                header = line.strip('\n')
                fasta_pairs[header] = ''
            else:
                fasta_pairs[header] += line.strip('\n')
        # return dict of fasta entries
        return fasta_pairs
    except IOError:
        print("File \"{}\" could not be opened.".format(file_name))

In [9]:
with open('intersection.pickle', 'rb') as handle:
    novel_intersect = pickle.load(handle)

In [10]:
names=["tama_augustus","tama_liftoff"]
runs=["../01_preparation/all_runs/tama_augustus_sqanti.pkl","../01_preparation/all_runs/tama_liftoff_sqanti.pkl"]

refs=pd.DataFrame()
for name, run in zip(names,runs):
    # load full df
    ref=pd.read_pickle(f"{run}")
    
    # add new column
    ref["cat_assign"]="None"
    # and fill new colum based on 4 categories
    ref.loc[(ref["structural_category"] != "antisense") & (ref["structural_category"] != "fusion") & (ref["structural_category"] != "intergenic"), "cat_assign"]="one2one"
    ref.loc[(ref["structural_category"] == "antisense"), "cat_assign"]="antisense"
    ref.loc[(ref["structural_category"] == "fusion"), "cat_assign"]="fusion"
    ref.loc[(ref["structural_category"] == "intergenic") | (ref["structural_category"] == "genic_intron"), "cat_assign"]="novel"
    
    # in some cases, transcripts were reported as one2one even though the reads are spanning two genes.
    # In these cases, we have "fusion reads", but splicing results in perfect cds for one gene.
    # filter them and add replace the assignment
    ref["association"] = ref["associated_gene"].str.split("_").str.len()
    ref.loc[(ref["cat_assign"]=="one2one")&(ref["association"]>1), "cat_assign"] = "fusion"
    
    ref = ref.loc[(ref["cat_assign"]=="antisense")|(ref["cat_assign"]=="novel")]    
    ref["run"]=name
    refs=pd.concat([refs,ref])

In [11]:
blast_df=pd.DataFrame()
for acc in tqdm(refs["accession"].unique().tolist()):
    acc_df={}
    ### Isoforms input
    # novel isoforms
    acc_df["novel_iso"]=len(novel_intersect[f"{acc}"])
    # novel isoforms with transdecoder hits
    fasta = read(f"../../output/v2/collapse_tama/{acc}_transdecoder/{acc}_transcripts.fasta.transdecoder.pep")
    keys=[]
    for k in fasta.keys():
        keys.append(k.replace(">","").split(".")[0] +"."+ k.replace(">","").split(".")[1])
    c=0
    for i in novel_intersect[f"{acc}"]:
        if i in set(keys):
            c+=1
    acc_df["novel_iso_trans"]=c
    # n transdecoder queries
    fasta_sub = read(f"input/{acc}_novel.prot.fasta")
    acc_df["trans_query"] = len(fasta_sub.keys())
    
    ### Blast
    blast=pd.read_csv(f"output/{acc}_blastp.txt", sep="\t", names=["query_id","subject_id","pct_identity","aln_length","n_of_mismatches", "gap_openings"," q_start", "q_end" ,"s_start","s_end","e_value","bit_score"])
    blast["isoform"]=blast["query_id"].str.split(".").str[0] + "." + blast["query_id"].str.split(".").str[1]
    # n transdecoder queries with hit
    acc_df["trans_hits"]=len(blast["query_id"].unique())
    # n isoforms with hit
    acc_df["iso_hits"]=len(blast["isoform"].unique())
    
    ### Filter blast
    f_blast = blast.loc[blast["pct_identity"]>90]
    # n transdecoder queries with hit
    acc_df["trans_hits_90"]=len(f_blast["query_id"].unique())
    # n isoforms with hit
    acc_df["iso_hits_90"]=len(f_blast["isoform"].unique())
    
    ### Information from ref
    # get best hits
    best_query=f_blast.sort_values(by=["pct_identity","e_value","bit_score"], ascending=False).drop_duplicates(subset="query_id", keep="first")
    best_query=best_query["isoform"].unique()
    # ref to check proteins against
    at_ref=refs.loc[(refs["accession"]==f"{acc}")&(refs["run"]=="tama_augustus")]
    
    #subset via isosform
    acc_df["antisense"]=at_ref.loc[at_ref["isoform"].isin(best_query)]["cat_assign"].value_counts()["antisense"]
    try:
        acc_df["novel"]=at_ref.loc[at_ref["isoform"].isin(best_query)]["cat_assign"].value_counts()["novel"]
    except:
        acc_df["novel"]=0
    
    # n coding
    acc_df["coding"]=len(at_ref.loc[(at_ref["isoform"].isin(best_query))&(at_ref["Sqanti_ORF_seq"].notna())])
    
    # combine
    acc_df["accession"]=acc
    
    acc_df=pd.DataFrame([acc_df])
    blast_df=pd.concat([blast_df,acc_df])

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [03:13<00:00, 10.76s/it]


In [12]:
blast_df

,novel_iso,novel_iso_trans,trans_query,trans_hits,iso_hits,trans_hits_90,iso_hits_90,antisense,novel,coding,accession
0,547,310,1080,1069,786,744,586,576,10,188,at6923
0,408,223,860,848,595,550,419,405,14,132,at6929
0,391,239,837,825,641,636,503,493,10,131,at7143
0,359,191,680,665,471,449,341,339,2,115,at8285
0,452,253,715,699,559,458,381,367,14,107,at9104
0,492,261,831,817,584,535,433,424,9,151,at9336
0,565,301,1221,1188,814,590,496,488,8,167,at9503
0,771,378,1261,1226,922,923,725,718,7,225,at9578
0,569,259,896,877,635,712,518,480,38,168,at9744
0,519,243,742,728,556,470,363,354,9,139,at9762


In [13]:
blast_df.to_csv("blast_hit_development.tsv", sep ="\t")